In [38]:
import yfinance as yf
from src.main import str_num
import numpy as np
t = yf.Ticker("AAPL")

cf  = t.cash_flow       # cash flow statement (annual)
bal = t.balance_sheet   # balance sheet
info = t.info           # dict of market data: price, market cap, shares, etc.  

In [39]:
ocf   = cf.loc["Operating Cash Flow"]
capex = cf.loc["Capital Expenditure"]

# CapEx is stored as a negative number in yfinance, so addition is correct
fcf = ocf + capex  # = OCF - |CapEx|
fcf = fcf.sort_index()  # put years in chronological order
weights = [0.2, 0.3, 0.5]  # oldest → newest
base_fcf = np.average(fcf.dropna().iloc[-3:], weights=weights)
fcf

2021-09-30             NaN
2022-09-30    1.114430e+11
2023-09-30    9.958400e+10
2024-09-30    1.088070e+11
2025-09-30    9.876700e+10
dtype: float64

In [40]:
# CAPM: cost of equity
Rf   = 0.043   # risk-free rate (10-yr Treasury)
ERP  = 0.045   # equity risk premium (Damodaran estimate)
beta = t.info.get("beta", 1.0)    # AAPL beta
print("Beta:", beta)
Ke = Rf + beta * ERP   # ~11.1%

# Capital structure weights
E = info["marketCap"]
D = bal.loc["Total Debt"].iloc[0]   # most recent year
V = E + D

# After-tax cost of debt
Kd  = 0.032
tax = 0.154

WACC = (E/V) * Ke + (D/V) * Kd * (1 - tax)

print("Discounted value: ", Ke)
print("Value:", str_num(V))
print("WACC: ", WACC)

Beta: 1.109
Discounted value:  0.09290499999999999
Value: 4_065_060_870_720.0
WACC:  0.09130726592231823


In [41]:
forecasted = []
fcf = base_fcf


# Pull it programmatically
growth_estimates = t.growth_estimates

stage1_growth = growth_estimates.loc["+1y", "stockTrend"] * 0.85
stage2_growth = stage1_growth * 0.6

for year in range(1, 8):
    growth = stage1_growth if year <= 4 else stage2_growth
    fcf = fcf * (1 + growth)
    forecasted.append(fcf)
    
g = 0.025

if WACC <= g:
    raise ValueError(f"WACC ({WACC:.2%}) must be greater than terminal growth ({g:.2%})")

TV = forecasted[-1] * (1 + g) / (WACC - g)

print("Terminal Value:", str_num(TV))

Terminal Value: 2_541_213_199_965.889


In [42]:
pv_fcfs = [fcf / (1 + WACC) ** (i + 1) for i, fcf in enumerate(forecasted)]
pv_tv   = TV / (1 + WACC) ** 7

enterprise_value = sum(pv_fcfs) + pv_tv
# It would cost this much to buy the whole company
print(f"Enterprise Value: ", str_num(enterprise_value))

Enterprise Value:  2_058_671_006_059.0488


In [43]:
cash  = bal.loc["Cash And Cash Equivalents"].iloc[0]
debt  = bal.loc["Total Debt"].iloc[0]
shares = info["sharesOutstanding"]

equity_value  = enterprise_value - debt + cash
intrinsic_price = equity_value / shares

print("Intrinsic price: ", intrinsic_price)

Intrinsic price:  135.95320295692628
